# UP2.6 · Pandas IV: agrupar y combinar

Ejemplos de la presentación, en el mismo orden. Este notebook va en la raíz del proyecto, junto a la carpeta `datos/`.

Ejecuta las celdas de una en una, de arriba abajo: cada una usa variables de las anteriores. Cada sección termina con una tabla de referencia.

In [1]:
import timeit
from pathlib import Path

import pandas as pd

pisos: pd.DataFrame = pd.read_csv(Path("datos") / "pisos.csv")

## 1. Agrupar

In [2]:
por_barrio: pd.Series = pisos.groupby("barrio")["precio"].mean().round()
print(por_barrio.sort_values(ascending=False).head(4))

barrio
Levante           1562.0
Rincón de Loix    1451.0
Centro            1404.0
La Cala           1399.0
Name: precio, dtype: float64


In [3]:
resumen: pd.DataFrame = pisos.groupby("tipo")["precio"].agg(
    ["count", "median", "max"]
)
print(resumen)

         count  median   max
tipo                        
dúplex      41  2040.0  3930
estudio     95   660.0  1440
piso       411  1170.0  3110
ático       53  1640.0  3260


In [4]:
por_tipo: pd.DataFrame = pisos.groupby("tipo").agg(
    anuncios=("id", "count"),
    precio_mediano=("precio", "median"),
    metros_medios=("metros", "mean"),
)
print(por_tipo.round(1))

         anuncios  precio_mediano  metros_medios
tipo                                            
dúplex         41          2040.0          122.9
estudio        95           660.0           36.2
piso          411          1170.0           84.3
ático          53          1640.0           92.6


In [5]:
grupo = pisos.groupby("tipo")
print(grupo.size().tolist())
print(grupo["planta"].count().tolist())

[41, 95, 411, 53]
[41, 92, 396, 52]


In [6]:
dos: pd.Series = (
    pisos.groupby(["tipo", "ascensor"])["precio"].median()
)
print(dos.head(4))

tipo     ascensor
dúplex   False       1510.0
         True        2065.0
estudio  False        535.0
         True         670.0
Name: precio, dtype: float64


In [7]:
tabla: pd.DataFrame = dos.reset_index()
print(tabla.head(4))

      tipo  ascensor  precio
0   dúplex     False  1510.0
1   dúplex      True  2065.0
2  estudio     False   535.0
3  estudio      True   670.0


### Referencia: agrupar

| Código | Qué hace |
|---|---|
| `df.groupby("col")["valor"].mean()` | Un resultado por grupo: la media de `valor` en cada valor de `col` |
| `.agg(["count", "median", "max"])` | Varias agregaciones a la vez: una columna por agregación |
| `.agg(nombre=("columna", "función"))` | Cada resultado con el nombre que se elija |
| `.size()` | Cuántas filas tiene cada grupo |
| `["col"].count()` | Cuántos valores no vacíos tiene la columna en cada grupo |
| `df.groupby(["a", "b"])` | Un grupo por cada combinación de valores de `a` y `b` |
| `.reset_index()` | Pasa las etiquetas de los grupos a columnas: vuelve a ser una tabla normal |

**Las agregaciones de siempre:** `count`, `sum`, `mean`, `median`, `min`, `max`, `std`, `nunique`.

## 2. Tablas dinámicas

In [8]:
cruce: pd.DataFrame = pisos.pivot_table(
    index="barrio",
    columns="tipo",
    values="precio",
    aggfunc="median",
)
print(cruce.head(5))

tipo           dúplex  estudio    piso   ático
barrio                                        
Casco Antiguo  1900.0    640.0  1150.0  1710.0
Centro         2065.0    660.0  1325.0  1980.0
El Murtal      1355.0    480.0  1040.0  1070.0
Els Tolls      1650.0    415.0   910.0  1640.0
Foietes        1620.0    495.0  1030.0  1140.0


In [9]:
cuantos: pd.DataFrame = pisos.pivot_table(
    index="tipo",
    columns="ascensor",
    values="id",
    aggfunc="count",
)
print(cuantos)

ascensor  False  True 
tipo                  
dúplex        9     32
estudio      22     73
piso        104    307
ático         4     49


### Referencia: tablas dinámicas

| Código | Qué hace |
|---|---|
| `df.pivot_table(index="a", columns="b", values="v", aggfunc="median")` | Una fila por valor de `a`, una columna por valor de `b`, y en cada celda la mediana de `v` |
| `aggfunc="count"` | En cada celda, cuántas filas hay |
| `fill_value=0` | Pone 0 en las combinaciones que no tienen ninguna fila, en lugar de `NaN` |

Es lo mismo que `groupby(["a", "b"])` con la segunda columna puesta en horizontal.

## 3. Combinar tablas

In [10]:
barrios: pd.DataFrame = pd.read_csv(Path("datos") / "barrios.csv")
print(barrios.head(4))

           barrio     zona  distancia_playa_km
0  Rincón de Loix  Levante                 0.3
1         Levante  Levante                 0.2
2          Centro   Centro                 0.5
3   Casco Antiguo   Centro                 0.2


In [11]:
con_zona: pd.DataFrame = pisos.merge(barrios, on="barrio")
print(pisos.shape, con_zona.shape)
print(con_zona[["barrio", "precio", "zona"]].head(3))

(600, 14) (600, 16)
     barrio  precio      zona
0   Levante    1830   Levante
1   Foietes     570  Interior
2  Poniente    1040  Poniente


In [12]:
print(con_zona.groupby("zona")["precio"].median())

zona
Centro      1330.0
Interior     930.0
Levante     1420.0
Poniente    1335.0
Name: precio, dtype: float64


In [13]:
playa: pd.DataFrame = barrios[barrios["zona"] != "Interior"]
solo_playa: pd.DataFrame = pisos.merge(playa, on="barrio")
todos: pd.DataFrame = pisos.merge(playa, on="barrio", how="left")
print(len(solo_playa), len(todos))
print(todos[["barrio", "zona"]].tail(3))

381 600
              barrio     zona
597    Casco Antiguo   Centro
598  Salt de l'Aigua      NaN
599   Rincón de Loix  Levante


In [14]:
primera: pd.DataFrame = pisos[pisos["fecha"] < "2026-05-01"]
segunda: pd.DataFrame = pisos[pisos["fecha"] >= "2026-05-01"]
juntas: pd.DataFrame = pd.concat([primera, segunda])
print(len(primera), len(segunda), len(juntas))

251 349 600


### Referencia: combinar tablas

| Código | Qué hace |
|---|---|
| `a.merge(b, on="col")` | Une las filas de `a` y `b` que tienen el mismo valor en `col`. Solo quedan las que tienen pareja |
| `how="left"` | Quedan todas las filas de `a`; las que no tienen pareja reciben `NaN` en las columnas de `b` |
| `how="inner"` | Solo las filas con pareja (es el valor por defecto) |
| `left_on="x", right_on="y"` | Cuando la columna común se llama distinto en cada tabla |
| `pd.concat([a, b])` | Pone `b` debajo de `a`. Las dos deben tener las mismas columnas |
| `pd.concat([a, b], ignore_index=True)` | Además, numera las filas de nuevo desde 0 |

**Comprueba siempre el tamaño** antes y después de unir: si una tabla tiene filas repetidas en la columna común, `merge` multiplica las filas.

## 4. Recorrer filas

In [15]:
for etiqueta, fila in pisos.head(3).iterrows():
    print(etiqueta, fila["barrio"], fila["precio"])

0 Levante 1830
1 Foietes 570
2 Poniente 1040


In [16]:
for anuncio in pisos.head(3).itertuples():
    print(f"{anuncio.barrio}: {anuncio.metros} m² por {anuncio.precio} €")

Levante: 115 m² por 1830 €
Foietes: 35 m² por 570 €
Poniente: 70 m² por 1040 €


In [17]:
def media_con_bucle(tabla: pd.DataFrame) -> dict[str, float]:
    suma: dict[str, float] = {}
    cuenta: dict[str, int] = {}
    for _, fila in tabla.iterrows():
        barrio: str = fila["barrio"]
        suma[barrio] = suma.get(barrio, 0) + fila["precio"]
        cuenta[barrio] = cuenta.get(barrio, 0) + 1
    return {b: suma[b] / cuenta[b] for b in suma}

def media_con_groupby(tabla: pd.DataFrame) -> pd.Series:
    return tabla.groupby("barrio")["precio"].mean()

t_bucle: float = timeit.timeit(lambda: media_con_bucle(pisos), number=5)
t_groupby: float = timeit.timeit(lambda: media_con_groupby(pisos), number=5)
print(f"groupby es {t_bucle / t_groupby:.0f} veces más rápido")

groupby es 33 veces más rápido


### Referencia: recorrer filas

| Código | Qué da en cada vuelta |
|---|---|
| `for etiqueta, fila in df.iterrows():` | La etiqueta y la fila como Series: `fila["barrio"]` |
| `for fila in df.itertuples():` | La fila como tupla con nombres: `fila.barrio`. Es más rápido que `iterrows` |

**Cuándo recorrer filas:** para hacer algo con cada fila que no es un cálculo: imprimirla, escribir un mensaje o llamar a una API por cada una.
**Cuándo no:** para calcular. Una operación con columnas, `groupby` o `np.where` hacen lo mismo en una línea y mucho más rápido.